In [1]:
from spatialx_transform.transforms import Transformation
from spatialx_transform.point import Point
import cv2 as cv
import numpy as np
import math

In [2]:
img = cv.imread("input.png")

[ WARN:0@0.026] global loadsave.cpp:278 findDecoder imread_('input.png'): can't open/read file: check file path/integrity


In [3]:
tf = Transformation.model_validate(
    {
        "transformation_type": "composed",
        "params": {
            "transforms": [
                {
                    "transformation_type": "affine",
                    "params": {
                        "A": [[0.7071, -0.7071], [0.7071, 0.7071]],
                        "b": [0, 0],
                    },
                },
                {
                    "transformation_type": "square",
                    "params": None,
                },
            ],
        },
    }
)

In [4]:
from spatialx_transform.warp import warp_transform

In [5]:
def draw_mesh(img, tf: Transformation, dx: np.int32, dy: np.int32, scale: np.float32):
    """Draw grid points and triangle mesh on input image and output canvas."""
    H_src = img.shape[0]
    W_src = img.shape[1]

    # --- Rebuild grid (same logic as warp_transform) ---
    offsetX, offsetY = 1e9, 1e9
    maxX, maxY = -1e9, -1e9

    for i in [0, H_src - 1]:
        for j in range(W_src):
            fw_point = tf.transform(Point([j, i]))
            px = fw_point.x * scale
            py = fw_point.y * scale
            offsetX = min(offsetX, py)
            offsetY = min(offsetY, px)
            maxX = max(maxX, py)
            maxY = max(maxY, px)
    for i in range(H_src):
        for j in [0, W_src - 1]:
            fw_point = tf.transform(Point([j, i]))
            px = fw_point.x * scale
            py = fw_point.y * scale
            offsetX = min(offsetX, py)
            offsetY = min(offsetY, px)
            maxX = max(maxX, py)
            maxY = max(maxY, px)

    W_dst = np.int32(maxY - offsetY) + 5
    H_dst = np.int32(maxX - offsetX) + 5

    offsetX = math.floor(offsetX)
    offsetY = math.floor(offsetY)

    approximated_X = list(range(0, H_src, dx))
    approximated_Y = list(range(0, W_src, dy))

    if approximated_X[-1] != H_src - 1:
        approximated_X.append(H_src - 1)
    if approximated_Y[-1] != W_src - 1:
        approximated_Y.append(W_src - 1)

    nx, ny = len(approximated_X), len(approximated_Y)
    trans_point = np.zeros((nx, ny), dtype=Point)

    for i in range(nx):
        for j in range(ny):
            x = approximated_X[i]
            y = approximated_Y[j]
            trans_point[i, j] = tf.transform(Point([y, x]))

    # --- Build triangle pairs ---
    srcTriangle = []
    dstTriangle = []

    for i in range(nx - 1):
        for j in range(ny - 1):
            srcTriangle.append(
                [
                    Point([approximated_Y[j], approximated_X[i]]),
                    Point([approximated_Y[j + 1], approximated_X[i]]),
                    Point([approximated_Y[j], approximated_X[i + 1]]),
                ]
            )
            dstTriangle.append(
                [trans_point[i][j], trans_point[i][j + 1], trans_point[i + 1][j]]
            )
            srcTriangle.append(
                [
                    Point([approximated_Y[j + 1], approximated_X[i + 1]]),
                    Point([approximated_Y[j + 1], approximated_X[i]]),
                    Point([approximated_Y[j], approximated_X[i + 1]]),
                ]
            )
            dstTriangle.append(
                [
                    trans_point[i + 1][j + 1],
                    trans_point[i][j + 1],
                    trans_point[i + 1][j],
                ]
            )

    # --- Draw on input image ---
    img_vis = img.copy()
    if img_vis.ndim == 2:
        img_vis = cv.cvtColor(img_vis, cv.COLOR_GRAY2BGR)

    for i in range(nx):
        for j in range(ny):
            px = int(approximated_Y[j])
            py = int(approximated_X[i])
            cv.circle(img_vis, (px, py), 1, (0, 0, 255), -1)

    for tri in srcTriangle:
        pts = [(int(p.x), int(p.y)) for p in tri]
        for k in range(3):
            cv.line(img_vis, pts[k], pts[(k + 1) % 3], (0, 255, 0), 1)

    # --- Draw on output canvas ---
    out_vis = np.zeros((H_dst, W_dst, 3), dtype=np.uint8)

    for i in range(nx):
        for j in range(ny):
            tp = trans_point[i][j]
            px = int(tp.x * scale - offsetY)
            py = int(tp.y * scale - offsetX)
            if 0 <= px < W_dst and 0 <= py < H_dst:
                cv.circle(out_vis, (px, py), 1, (0, 0, 255), -1)

    for tri in dstTriangle:
        pts = []
        for p in tri:
            px = int(p.x * scale - offsetY)
            py = int(p.y * scale - offsetX)
            pts.append((px, py))
        for k in range(3):
            x1, y1 = pts[k]
            x2, y2 = pts[(k + 1) % 3]
            # Only draw if at least one endpoint is visible
            if (0 <= x1 < W_dst and 0 <= y1 < H_dst) or (
                0 <= x2 < W_dst and 0 <= y2 < H_dst
            ):
                cv.line(out_vis, (x1, y1), (x2, y2), (0, 255, 0), 1)

    # --- Display side by side ---
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    axes[0].imshow(cv.cvtColor(img_vis, cv.COLOR_BGR2RGB))
    axes[0].set_title(f"Input + Mesh ({nx}x{ny} grid, {len(srcTriangle)} triangles)")
    axes[0].axis("off")
    axes[1].imshow(cv.cvtColor(out_vis, cv.COLOR_BGR2RGB))
    axes[1].set_title(f"Output + Mesh ({W_dst}x{H_dst})")
    axes[1].axis("off")
    plt.tight_layout()
    plt.savefig("mesh_visualization.png", dpi=150)
    plt.show()

    print(f"Grid: {nx}x{ny}, Triangles: {len(srcTriangle)}, Output: {W_dst}x{H_dst}")

In [6]:
import json
import zarr
import matplotlib.pyplot as plt

IMG2_DIR = "/home/quan.lt/scratch/spatialx-transform/data_test/1780244765_9cd59a4efb4a4d1e890e920f8a65aa93"

# --- Load alignment transform from alignment_test.json ---
with open(f"{IMG2_DIR}/alignment_test5.json") as f:
    aln_data = json.load(f)
align_tf = Transformation.model_validate(aln_data["alignment"]["transformation"])
print(
    f"Loaded transform: {aln_data['alignment']['transformation']['transformation_type']}"
)
print(f"Alignment name: {aln_data['alignment_name']}")

# --- Load img2 at level 5 (32x downsampled) ---
z_img2 = zarr.open(f"{IMG2_DIR}/5", mode="r")
print(f"img2 level 5 shape: {z_img2.shape}")

Loaded transform: composed
Alignment name: Image alignment
img2 level 5 shape: (5, 2934, 1197)


In [7]:
# Sanity check: transform image center -> should land in img1 space
center = Point([z_img2.shape[2] // 2, z_img2.shape[1] // 2])
mapped = align_tf.transform(center)
print(f"Image center -> img1 coords: ({mapped.x:.1f}, {mapped.y:.1f})")

Image center -> img1 coords: (586.0, 1304.6)


In [8]:
# --- Input composite: stack 5 raw channels with color mapping (before warping) ---
CHANNEL_NAMES = [
    "AF488_PanCK",
    "ATTO532_CD3",
    "Dy605_Membrane",
    "AF647_CD45",
    "DAPI_DNA",
]
CHANNEL_COLORS = [
    [0, 255, 0],  # PanCK  -> green
    [255, 0, 0],  # CD3    -> red
    [0, 0, 255],  # Dy605  -> blue
    [255, 0, 255],  # CD45   -> magenta
    [255, 255, 255],  # DAPI   -> trắng/xám (nhân tế bào làm nền)
]
NUM_CHANNELS = z_img2.shape[0]

# input_composite = np.zeros((z_img2.shape[1], z_img2.shape[2], 3), dtype=np.float32)
# input_colored = []
# for ch in range(NUM_CHANNELS):
#     ch_data = z_img2[ch, :, :]
#     gray = ((ch_data - ch_data.min()) / max(1, ch_data.max() - ch_data.min())).astype(np.float32)
#     color = np.array(CHANNEL_COLORS[ch], dtype=np.float32)
#     input_colored.append((gray[:, :, None] * color[None, None, :]).astype(np.uint8))
#     input_composite += gray[:, :, None] * color[None, None, :]

# input_composite = np.clip(input_composite, 0, 255).astype(np.uint8)
# cv.imwrite('input_composite.png', input_composite)

# fig, axes = plt.subplots(2, 3, figsize=(18, 12))
# for i in range(NUM_CHANNELS):
#     row, col = divmod(i, 3)
#     axes[row][col].imshow(input_colored[i])
#     axes[row][col].set_title(CHANNEL_NAMES[i])
#     axes[row][col].axis('off')
# axes[1][2].imshow(input_composite)
# axes[1][2].set_title('Input Composite (all 5)')
# axes[1][2].axis('off')
# plt.tight_layout()
# plt.savefig('input_composite_overview.png', dpi=150)
# plt.show()

In [9]:
# Run warp_transform on all 5 channels
warped_channels = []
for ch in range(NUM_CHANNELS):
    print(f"Warping channel {ch}: {CHANNEL_NAMES[ch]} ...")
    ch_data = z_img2[ch, :, :]
    ch_u8 = (
        (ch_data - ch_data.min()) / max(1, ch_data.max() - ch_data.min()) * 255
    ).astype(np.uint8)
    result = warp_transform(ch_u8, align_tf, 8, 8, 1, verbose=True)
    warped_channels.append(result)
    cv.imwrite(f"output_ch{ch}_{CHANNEL_NAMES[ch]}.png", result)
    print(f"  -> shape: {result.shape}")
print(f"Done. {NUM_CHANNELS} channels warped.")

# Visualize all 5 warped channels
# fig, axes = plt.subplots(1, NUM_CHANNELS, figsize=(6 * NUM_CHANNELS, 6))
# for i in range(NUM_CHANNELS):
#     axes[i].imshow(warped_channels[i])
#     axes[i].set_title(CHANNEL_NAMES[i])
#     axes[i].axis('off')
# plt.tight_layout()
# plt.savefig('all_channels_warped.png', dpi=150)
# plt.show()

Warping channel 0: AF488_PanCK ...


IndexError: tuple index out of range

In [ ]:
# Draw grid + triangle mesh visualization (channel 0 as background)
ch0_data = z_img2[0, :, :]
ch0_u8 = (
    (ch0_data - ch0_data.min()) / max(1, ch0_data.max() - ch0_data.min()) * 255
).astype(np.uint8)
ch0_rgb = np.stack([ch0_u8] * 3, axis=-1)
draw_mesh(ch0_rgb, align_tf, 8, 8, 1)

In [ ]:
# --- Output composite: stack 5 warped channels with color mapping (additive sum) ---
composite = np.zeros_like(warped_channels[0], dtype=np.float32)
colored_channels = []
for ch_idx, warped in enumerate(warped_channels):
    gray = cv.cvtColor(warped, cv.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    color = np.array(CHANNEL_COLORS[ch_idx], dtype=np.float32)
    colored = (gray[:, :, None] * color[None, None, :]).astype(np.uint8)
    colored_channels.append(colored)
    composite += gray[:, :, None] * color[None, None, :]

composite = np.clip(composite, 0, 255).astype(np.uint8)
cv.imwrite("composite.png", cv.cvtColor(composite, cv.COLOR_RGB2BGR))

# Visualize: 5 individual colored channels + composite
# fig, axes = plt.subplots(2, 3, figsize=(18, 12))
# for i in range(NUM_CHANNELS):
#     row, col = divmod(i, 3)
#     axes[row][col].imshow(colored_channels[i])
#     axes[row][col].set_title(CHANNEL_NAMES[i])
#     axes[row][col].axis('off')
# axes[1][2].imshow(composite)
# axes[1][2].set_title('Output Composite (all 5)')
# axes[1][2].axis('off')
# plt.tight_layout()
# plt.savefig('composite_overview.png', dpi=150)
# plt.show()